In [1]:
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu126

Looking in indexes: https://download.pytorch.org/whl/cu126


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.autograd import grad
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set()



<br/>
<div>
    <img src="https://upload.wikimedia.org/wikipedia/commons/thumb/c/cc/Comparison_image_neural_networks.svg/1920px-Comparison_image_neural_networks.svg.png" width="800"/>
</div>

Создадим сеть AlexNet согласно этой схеме

In [3]:
'''
class AlexNet(nn.Module):
    def __init__(self, num_classes=1000):
        super().__init__()

        self.layers = nn.Sequential(
            #первая свертка, 3 - кол-во цветов(RGB), 96 признаков
            #скользящее окно 11 на 11, 4 - перемещение на 4 пикселя за один шаг
            nn.Conv2d(3, 96, kernel_size=11, stride=4, padding=0),
            nn.ReLU(inplace=True), #ускоряет процесс обучения

            #скользящее окно 3 на 3, перемещение на 2 пикселя за шаг
            nn.MaxPool2d(kernel_size=3, stride=2),

            #по аналогии делаем с отсавшейся частью схемы

            #96 - кол-во выходных каналов для прошлого слоя -> кол-во входных для этого
            nn.Conv2d(96, 256, kernel_size=5, padding=2),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(kernel_size=3, stride=2),

            nn.Conv2d(256, 384, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),

            nn.Conv2d(384, 384, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),

            nn.Conv2d(384, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(kernel_size=3, stride=2),

            #для переход от свертки до полносвязной части
            nn.Flatten(),

            #полносвязный слой
            nn.Linear(256 * 5 * 5, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.5),

            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.5),

            nn.Linear(4096, num_classes)
        )

    def forward(self, x):
      return self.layers(x)

model = AlexNet(num_classes=1000)
print(model)
'''

'\nclass AlexNet(nn.Module):\n    def __init__(self, num_classes=1000):\n        super().__init__()\n\n        self.layers = nn.Sequential(\n            #первая свертка, 3 - кол-во цветов(RGB), 96 признаков\n            #скользящее окно 11 на 11, 4 - перемещение на 4 пикселя за один шаг\n            nn.Conv2d(3, 96, kernel_size=11, stride=4, padding=0),\n            nn.ReLU(inplace=True), #ускоряет процесс обучения\n\n            #скользящее окно 3 на 3, перемещение на 2 пикселя за шаг\n            nn.MaxPool2d(kernel_size=3, stride=2),\n\n            #по аналогии делаем с отсавшейся частью схемы\n\n            #96 - кол-во выходных каналов для прошлого слоя -> кол-во входных для этого\n            nn.Conv2d(96, 256, kernel_size=5, padding=2),\n            nn.ReLU(inplace=True),\n\n            nn.MaxPool2d(kernel_size=3, stride=2),\n\n            nn.Conv2d(256, 384, kernel_size=3, padding=1),\n            nn.ReLU(inplace=True),\n\n            nn.Conv2d(384, 384, kernel_size=3, padding=

Будем использовать для обучения датасет CIFAR-10

In [4]:
import torchvision
import torchvision.transforms as transforms

In [5]:
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))])

train = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
test = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

train_dataloader = DataLoader(train, batch_size=128, shuffle=True, num_workers=2)
test_dataloader = DataLoader(test, batch_size=128, shuffle=False, num_workers=2)

100%|██████████| 170M/170M [00:13<00:00, 12.4MB/s]


посмотрим распределение изображений по классам

In [6]:
from collections import Counter

def class_distribution(dataloader, class_names):
    all_labels = []
    for _, labels in dataloader:
        all_labels.extend(labels.tolist())

    class_counts = Counter(all_labels)

    for class_idx, count in class_counts.items():
        class_name = class_names[class_idx]
        print(f"{class_name}: {count}")

    total_images = sum(class_counts.values())
    print(f"Всего изображений {total_images}")

cifar10_classes = ['airplane', 'automobile', 'bird', 'cat', 'deer',
                   'dog', 'frog', 'horse', 'ship', 'truck']

train_distribution = class_distribution(train_dataloader, cifar10_classes)

ship: 5000
automobile: 5000
truck: 5000
frog: 5000
bird: 5000
horse: 5000
airplane: 5000
cat: 5000
deer: 5000
dog: 5000
Всего изображений 50000


поменяем модель под набор данных который будем дальше использовать

In [7]:
class AlexNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(64, 192, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(192, 384, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),

            nn.Conv2d(384, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),

            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        self.classifier = nn.Sequential(
            nn.Dropout(),
            nn.Linear(256 * 4 * 4, 1024),
            nn.ReLU(inplace=True),
            nn.Dropout(),
            nn.Linear(1024, 512),
            nn.ReLU(inplace=True),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

In [10]:
#нужно 10 классов
model = AlexNet(num_classes=10)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=15, gamma=0.5)

def accuracy(model, dataloader, device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x, y in dataloader:
            x, y = x.to(device), y.to(device)
            outputs = model(x)
            _, predicted = torch.max(outputs, 1)
            total += y.size(0) #суммируем кол-во элементов в текущем батче
            correct += (predicted == y).sum().item() #суммируем правильные предсказания
    return 100 * correct / total

num_epochs = 20
best_acc = 0

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for batch_idx, (x, y) in enumerate(train_dataloader):
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        outputs = model(x)
        loss = F.cross_entropy(outputs, y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    epoch_loss = running_loss / len(train_dataloader) #ошибка в эпохе
    train_acc = accuracy(model, train_dataloader, device) #точночть на обучающей
    test_acc = accuracy(model, test_dataloader, device) #точность на тестовой

    scheduler.step()

    print(f'Epoch {epoch+1}/{num_epochs}:')
    print(f'Loss: {epoch_loss:.4f}, Train Acc: {train_acc:.2f}%, Test Acc: {test_acc:.2f}%')

Epoch 1/20:
Loss: 1.6716, Train Acc: 52.25%, Test Acc: 51.58%
Epoch 2/20:
Loss: 1.2214, Train Acc: 62.99%, Test Acc: 62.02%
Epoch 3/20:
Loss: 1.0243, Train Acc: 69.15%, Test Acc: 66.72%
Epoch 4/20:
Loss: 0.8898, Train Acc: 73.41%, Test Acc: 70.21%
Epoch 5/20:
Loss: 0.7878, Train Acc: 77.73%, Test Acc: 73.72%
Epoch 6/20:
Loss: 0.7185, Train Acc: 80.19%, Test Acc: 74.56%
Epoch 7/20:
Loss: 0.6506, Train Acc: 80.02%, Test Acc: 74.34%
Epoch 8/20:
Loss: 0.6054, Train Acc: 83.25%, Test Acc: 76.41%
Epoch 9/20:
Loss: 0.5606, Train Acc: 84.47%, Test Acc: 76.79%
Epoch 10/20:
Loss: 0.5212, Train Acc: 86.54%, Test Acc: 77.70%
Epoch 11/20:
Loss: 0.4929, Train Acc: 87.06%, Test Acc: 77.00%
Epoch 12/20:
Loss: 0.4668, Train Acc: 88.32%, Test Acc: 77.77%
Epoch 13/20:
Loss: 0.4392, Train Acc: 88.97%, Test Acc: 78.21%
Epoch 14/20:
Loss: 0.4128, Train Acc: 90.31%, Test Acc: 78.47%
Epoch 15/20:
Loss: 0.3909, Train Acc: 89.65%, Test Acc: 77.51%
Epoch 16/20:
Loss: 0.2806, Train Acc: 94.73%, Test Acc: 79.82%
E

In [11]:
def class_accuracy(model, dataloader, class_names, device):
    model.eval()

    class_correct = [0] * len(class_names) #список для кол-ва верных предсказаний
    class_total = [0] * len(class_names) #список для общего кол-ва примеров

    with torch.no_grad():
        for x, y in dataloader:
            x, y = x.to(device), y.to(device)
            outputs = model(x)
            _, predicted = torch.max(outputs, 1)
            #рассматриваем результаты по каждому изображению
            for i in range(len(y)):
                label = y[i].item()
                class_total[label] += 1
                if predicted[i].item() == label:
                    class_correct[label] += 1

    total_correct = 0
    total_samples = 0

    for i in range(len(class_names)):
        if class_total[i] > 0:
            accuracy = 100 * class_correct[i] / class_total[i]
            total_correct += class_correct[i]
            total_samples += class_total[i]
            print(f"{class_names[i]}: {accuracy:6.2f}")

    overall_accuracy = 100 * total_correct / total_samples if total_samples > 0 else 0
    print(f"{'Точность по всем классам'}: {overall_accuracy:6.2f}")

    return overall_accuracy

print("Точность на обучающих:")
train_overall_acc = class_accuracy(model, train_dataloader, cifar10_classes, device)

print("\nТочность на тестовых:")
test_overall_acc = class_accuracy(model, test_dataloader, cifar10_classes, device)

Точность на обучающих:
airplane:  98.96
automobile:  99.22
bird:  97.66
cat:  89.00
deer:  97.76
dog:  87.82
frog:  98.00
horse:  98.40
ship:  99.14
truck:  98.42
Точность по всем классам:  96.44

Точность на тестовых:
airplane:  85.20
automobile:  90.10
bird:  75.30
cat:  56.80
deer:  79.00
dog:  62.20
frog:  84.60
horse:  82.30
ship:  89.20
truck:  87.40
Точность по всем классам:  79.21
